In [1]:
import pandas as pd
esol = pd.read_csv('../data/esol_v2.csv')

In [2]:
print(list(esol.columns))
esol.head()

['Compound ID', 'ESOL predicted log solubility in mols per litre', 'Minimum Degree', 'Molecular Weight', 'Number of H-Bond Donors', 'Number of Rings', 'Number of Rotatable Bonds', 'Polar Surface Area', 'measured log solubility in mols per litre', 'smiles']


,Compound ID,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre,smiles
0,Amigdalin,-0.974,1,457.432,7,3,7,202.32,-0.77,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...
1,Fenfuram,-2.885,1,201.225,1,2,2,42.24,-3.30,Cc1occc1C(=O)Nc2ccccc2
2,citral,-2.579,1,152.237,0,0,4,17.07,-2.06,CC(C)=CCCC(C)=CC(=O)
3,Picene,-6.618,2,278.354,0,5,0,0.00,-7.87,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43
4,Thiophene,-2.232,2,84.143,0,1,0,0.00,-1.33,c1ccsc1


In [3]:
import numpy as np
import sys
sys.path.append('../')
from util.dl_util import *

esol_smiles = list(esol['smiles'])
esol_smiles = np.array(esol_smiles)
esol_lable = esol['measured log solubility in mols per litre']
esol_lable = np.array(esol_lable)

from util.rdkit_util import *

valid_smiles , valid_maccs , valid_lable = get_maccs_key(esol_smiles, esol_lable)

vocab_size = generate_math_code_tokenizer(valid_smiles, encode_type='vocab_size')

In [4]:
print(esol_smiles[0])
vocab_size

OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)C(O)C3O 


33

In [5]:
esol_len_list = []

for smile in valid_smiles:
    esol_len_list += [len(smile)]

esol_max_len = max(esol_len_list)
esol_max_len

98

In [6]:
esol_smile_sequence = generate_math_code_tokenizer(valid_smiles)
print(len(esol_smile_sequence[0]))
print(esol_smile_sequence[0])

54
[5, 1, 1, 11, 5, 1, 3, 5, 1, 1, 8, 5, 1, 3, 5, 1, 3, 1, 24, 10, 4, 2, 6, 2, 2, 2, 2, 2, 6, 4, 1, 3, 5, 4, 1, 3, 5, 4, 1, 8, 5, 4, 1, 3, 5, 4, 1, 3, 5, 4, 1, 11, 5, 13]


In [7]:
from keras.preprocessing import sequence

esol_smile_sequence = sequence.pad_sequences(esol_smile_sequence, maxlen=esol_max_len)

print(len(esol_smile_sequence))
print(esol_smile_sequence) # padding 参数 前缀补0，和后缀补0的差异

1128
[[ 0  0  0 ... 11  5 13]
 [ 0  0  0 ...  2  2  8]
 [ 0  0  0 ...  7  5  4]
 ...
 [ 0  0  0 ...  4  5  1]
 [ 0  0  0 ...  1  4  1]
 [ 0  0  0 ...  6  1  9]]


In [8]:
X1 = esol_smile_sequence
X2 = valid_maccs
Y = valid_lable

In [ ]:
from util.chemixnet_model_util import *
from util.eval_util import *


y_test, y_pred = split_data_and_train_model(CNN_MODEL,X1,X2,Y, vocab_size, esol_max_len, 'evol_esol',batch_size=16, epochs=100)

CNN_MODEL


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 98, 128)        │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 96, 512)        │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 48, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 46, 256)        │       393,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 23, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 21, 128)        │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 8, 64)          │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 4, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 717,953 (2.74 MB)

 Trainable params: 717,953 (2.74 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 3.5143 - mae: 1.4323 - val_loss: 1.3794 - val_mae: 0.9256
Epoch 2/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 1.7767 - mae: 1.0631 - val_loss: 1.0740 - val_mae: 0.8194
Epoch 3/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 1.5264 - mae: 0.9826 - val_loss: 1.0138 - val_mae: 0.7784
Epoch 4/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 1.3547 - mae: 0.9216 - val_loss: 0.9493 - val_mae: 0.7766
Epoch 5/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 1.2093 - mae: 0.8671 - val_loss: 0.8931 - val_mae: 0.7621
Epoch 6/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 1.0744 - mae: 0.8123 - val_loss: 0.8661 - val_mae: 0.7482
Epoch 7/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.9704 - mae: 0.7628 - val_loss: 0.8351 - val_mae: 0.7307
Epoch 8/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.8871 - mae: 0.7211 - val_loss: 0.8959 - val_mae: 0.7596
Epoch 9/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - lo

: 